# 75 - Experimento MPNet + HDBSCAN

Este notebook agrupa descrições normalizadas de itens de supermercado usando embeddings do modelo `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` e HDBSCAN. A análise usa distância cosseno, não usa NCM nem rótulos humanos e trata ruído como resultado explícito do algoritmo. A amostra configurável mantém o custo do experimento controlado. A execução requer `sentence-transformers` no ambiente e pode baixar o modelo do Hugging Face no primeiro uso. Execute de cima para baixo em uma sessão limpa.


In [1]:
from sklearn.decomposition import TruncatedSVD
from sklearn import config_context
from hashlib import sha256
from datetime import datetime, timezone
from pathlib import Path
import logging
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from matplotlib.colors import ListedColormap

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook75")
pd.set_option("display.max_colwidth", 140)


## Configuracoes e comparabilidade

Controles comuns aos notebooks 70 a 76:

- Base completa `2025_NCMs_filtrados_semstopwords`, periodo 202501-202512.
- Ate 20.000 descricoes unicas para agrupamento, amostradas com semente 42 e ordenadas por descricao normalizada. Frequencia nao pondera o ajuste.
- Silhouette cosseno sobre uma amostra base fixa de ate 20.000 linhas, com semente 42. Ruido do algoritmo e excluido depois da amostragem; a quantidade efetivamente avaliada e registrada.
- Ranking exige silhouette definida e de 10 a 1.000 clusters efetivamente formados, sem contar ruido. Pesos: 0,75 para silhouette reescalada e 0,25 para cobertura do algoritmo.
- t-SNE 2D e 3D: mesmos ate 5.000 itens, semente 42, TruncatedSVD de ate 50 componentes, perplexidade 30, 1.000 iteracoes, init PCA (quando ha dimensoes suficientes), learning_rate auto e Barnes-Hut. As projecoes de representacoes diferentes sao exploratorias, nao um espaco comum de distancias.
- `comparison` em `run.json` registra os controles e o hash das descricoes/frequencias usadas no agrupamento.

MPNet: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`, batch_size 64, normalize_embeddings True e saida float32. Nao ha min_df ou analyzer no MPNet puro.

HDBSCAN: min_cluster_size 2/5/10/20, min_samples None/2/5/10, selecao eom, metric cosine, algorithm brute e n_jobs 1, como nos notebooks 74 e 76. Grade atual: 16 ajustes.


In [2]:
DATASET_NAME = "2025_NCMs_filtrados_semstopwords"

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei a raiz do repositório com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/período explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve começar com '{DATASET_NAME}-' para derivar o período: "
        f"{DATASET_PATH.name}"
    )

PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
ENCODE_BATCH_SIZE = 64
MAX_DESCRIPTIONS = 20_000
SAMPLE_SEEDS = [42]
MIN_CLUSTER_SIZE_VALUES = [2, 5, 10, 20]
MIN_SAMPLES_VALUES = [None, 2, 5, 10]
CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_N_JOBS = 1
SILHOUETTE_SAMPLE_SIZE = 20_000
OUTPUT_ROOT = PROJECT_ROOT / "reports/mpnet_hdbscan"
MODEL_ROOT = PROJECT_ROOT / "models/mpnet_hdbscan"


TSNE_MAX_POINTS = 5_000
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
TSNE_3D_N_COMPONENTS = 3

METRIC_RANDOM_STATE = 42

TSNE_SVD_COMPONENTS = 50

PAIRWISE_WORKING_MEMORY_MIB = 64

ASSIGNMENT_RUN_ID = None

MODEL_RUN_ID = None


### Configuração do ranking final

`MIN_CLUSTER_FOR_RANKING = 10` define o minimo do ranking. O intervalo ate `MAX_CLUSTERS_FOR_RANKING` inclui os dois extremos e nao altera os ajustes da grade. Em DBSCAN/HDBSCAN, a contagem exclui o grupo de ruido (-1).


In [3]:
# O limite filtra apenas o ranking e a seleção final; não altera os ajustes do HDBSCAN.
MAX_CLUSTERS_FOR_RANKING = 1000
MIN_CLUSTER_FOR_RANKING = 10

# Os pesos definem a contribuição relativa de silhouette e cobertura (1 - ruído).
SILHOUETTE_WEIGHT = 0.75
NOISE_WEIGHT = 0.25


## 2. Funções da análise

A preparação remove descrições vazias, mantém uma linha por descrição e valida que a frequência total não foi alterada. Cada configuração HDBSCAN de uma semente usa os mesmos embeddings MPNet.


In [4]:
def prepare_items(frame):
    """Limpa descrições, agrega duplicatas e preserva a frequência total."""
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = (
        prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY]
        .sum()
    )
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A preparação alterou a soma de frequência.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)

def summarize_clusters(items, labels):
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary

def score_configuration(matrix, labels, sample_size, random_state):
    """Avalia a mesma amostra de linhas, excluindo ruido somente apos amostrar."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_count = len(np.unique(labels[~noise_mask]))
    base_count = min(sample_size, len(labels))
    rng = np.random.default_rng(random_state)
    positions = np.sort(rng.choice(len(labels), size=base_count, replace=False))
    positions = positions[~noise_mask[positions]]
    sample_labels = labels[positions]
    sample_cluster_count = len(np.unique(sample_labels))
    score = np.nan
    actual_count = len(positions)
    if 2 <= sample_cluster_count < actual_count:
        with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
            score = float(silhouette_score(matrix[positions], sample_labels, metric="cosine"))
    else:
        actual_count = 0
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_base_sample_size": base_count,
        "silhouette_sample_size": actual_count,
        "silhouette_cosine_without_noise": score,
    }

def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state, n_components=2):
    """Usa a mesma amostra, SVD e parametros t-SNE em todas as representacoes."""
    point_count = features.shape[0]
    sample_count = min(point_count, max_points)
    if sample_count < 3 or n_components not in (2, 3):
        raise ValueError("t-SNE exige pelo menos tres itens e n_components 2 ou 3.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled = features[indices]
    dimensions = min(TSNE_SVD_COMPONENTS, sampled.shape[1] - 1, sample_count - 1)
    if dimensions >= 1:
        reduced = TruncatedSVD(n_components=dimensions, random_state=random_state).fit_transform(sampled)
    else:
        reduced = sampled.toarray() if hasattr(sampled, "toarray") else np.asarray(sampled)
        dimensions = reduced.shape[1]
    coordinates = TSNE(
        n_components=n_components,
        perplexity=min(float(perplexity), (sample_count - 1) / 3),
        init="pca" if dimensions >= n_components else "random",
        learning_rate="auto", max_iter=max_iter, random_state=random_state,
        method="barnes_hut",
    ).fit_transform(reduced)
    return indices, coordinates, dimensions


def save_tsne_plot(coordinates, labels, run_id, output_path):
    """Salva um PNG t-SNE colorido pelos clusters, com o ruído em cinza."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_ids = np.unique(labels[~noise_mask])
    if cluster_ids.size:
        palette = plt.get_cmap("tab20").colors
        cluster_cmap = ListedColormap([
            palette[index % len(palette)] for index in range(len(cluster_ids))
        ])
        cluster_codes = np.searchsorted(cluster_ids, labels[~noise_mask])
    else:
        cluster_cmap = None
        cluster_codes = np.empty(0, dtype=int)

    figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
    if noise_mask.any():
        axis.scatter(
            coordinates[noise_mask, 0], coordinates[noise_mask, 1],
            c="#b8b8b8", s=8, alpha=0.3, linewidths=0,
        )
    if cluster_ids.size:
        axis.scatter(
            coordinates[~noise_mask, 0], coordinates[~noise_mask, 1],
            c=cluster_codes, cmap=cluster_cmap, vmin=-0.5,
            vmax=len(cluster_ids) - 0.5, s=9, alpha=0.75, linewidths=0,
        )
    axis.set_title(
        f"t-SNE: {run_id}\n{len(cluster_ids)} clusters; "
        f"{int(noise_mask.sum()):,} descrições como ruído"
    )
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.text(
        0.01, 0.01,
        "Ruído em cinza; as cores se repetem a cada 20 clusters.",
        transform=axis.transAxes, fontsize=8, color="#555555",
    )
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


def save_tsne_3d_html(coordinates, labels, run_id, output_path):
    """Salva uma visualização t-SNE 3D interativa e autocontida em HTML."""
    points = [
        {"x": float(x), "y": float(y), "z": float(z), "label": int(label)}
        for (x, y, z), label in zip(coordinates, labels)
    ]
    data_json = json.dumps(points, separators=(",", ":"))
    title = str(run_id).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    page = r"""<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>t-SNE 3D: __TITLE__</title>
<style>
html, body { margin: 0; height: 100%; background: #f7f8fa; color: #202124; font: 14px system-ui, sans-serif; }
main { display: flex; flex-direction: column; height: 100%; }
header { padding: 12px 18px; background: white; border-bottom: 1px solid #ddd; }
h1 { margin: 0 0 4px; font-size: 17px; }
p { margin: 0; color: #555; }
canvas { display: block; width: 100%; flex: 1; min-height: 300px; cursor: grab; }
canvas:active { cursor: grabbing; }
</style>
</head>
<body>
<main>
<header><h1>Projeção t-SNE 3D: __TITLE__</h1><p>Arraste para girar; use a roda do mouse para aproximar. Ruído em cinza.</p></header>
<canvas id="plot"></canvas>
</main>
<script>
const points = __POINTS__;
const canvas = document.getElementById("plot");
const ctx = canvas.getContext("2d");
let yaw = 0.65, pitch = -0.35, zoom = 1, dragging = false, lastX = 0, lastY = 0;
const colors = ["#3366cc", "#dc3912", "#ff9900", "#109618", "#990099", "#0099c6", "#dd4477", "#66aa00", "#b82e2e", "#316395", "#994499", "#22aa99", "#aaaa11", "#6633cc", "#e67300", "#8b0707", "#651067", "#329262", "#5574a6", "#3b3eac"];
const bounds = ["x", "y", "z"].map(key => {
  const values = points.map(point => point[key]);
  const low = Math.min(...values), high = Math.max(...values);
  return { low, span: Math.max(high - low, 1e-9) };
});
function resize() {
  const ratio = window.devicePixelRatio || 1;
  canvas.width = Math.max(1, Math.floor(canvas.clientWidth * ratio));
  canvas.height = Math.max(1, Math.floor(canvas.clientHeight * ratio));
  ctx.setTransform(ratio, 0, 0, ratio, 0, 0);
  draw();
}
function draw() {
  const width = canvas.clientWidth, height = canvas.clientHeight;
  ctx.clearRect(0, 0, width, height);
  const scale = Math.min(width, height) * 0.7 * zoom;
  const cy = Math.cos(yaw), sy = Math.sin(yaw), cp = Math.cos(pitch), sp = Math.sin(pitch);
  const projected = points.map(point => {
    const x = ((point.x - bounds[0].low) / bounds[0].span - 0.5) * 2;
    const y = ((point.y - bounds[1].low) / bounds[1].span - 0.5) * 2;
    const z = ((point.z - bounds[2].low) / bounds[2].span - 0.5) * 2;
    const rx = x * cy - z * sy, rz = x * sy + z * cy;
    const ry = y * cp - rz * sp, depth = y * sp + rz * cp;
    return { x: width / 2 + rx * scale / 2, y: height / 2 - ry * scale / 2, depth, label: point.label };
  }).sort((a, b) => a.depth - b.depth);
  for (const point of projected) {
    ctx.globalAlpha = point.label === -1 ? 0.24 : 0.78;
    ctx.fillStyle = point.label === -1 ? "#999999" : colors[Math.abs(point.label) % colors.length];
    ctx.beginPath();
    ctx.arc(point.x, point.y, point.label === -1 ? 2 : 3, 0, Math.PI * 2);
    ctx.fill();
  }
  ctx.globalAlpha = 1;
}
canvas.addEventListener("pointerdown", event => { dragging = true; lastX = event.clientX; lastY = event.clientY; canvas.setPointerCapture(event.pointerId); });
canvas.addEventListener("pointerup", () => { dragging = false; });
canvas.addEventListener("pointermove", event => {
  if (!dragging) return;
  yaw += (event.clientX - lastX) * 0.008;
  pitch = Math.max(-1.5, Math.min(1.5, pitch + (event.clientY - lastY) * 0.008));
  lastX = event.clientX; lastY = event.clientY; draw();
});
canvas.addEventListener("wheel", event => { event.preventDefault(); zoom = Math.max(0.3, Math.min(5, zoom * (event.deltaY < 0 ? 1.1 : 0.9))); draw(); }, { passive: false });
window.addEventListener("resize", resize);
resize();
</script>
</body>
</html>"""
    page = page.replace("__TITLE__", title).replace("__POINTS__", data_json)
    output_path.write_text(page, encoding="utf-8")


def comparison_metadata(frame):
    """Registra controles comuns e a identidade exata da amostra de agrupamento."""
    sample = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    return {
        "dataset_name": DATASET_NAME,
        "training_description_count": len(sample),
        "training_sample_sha256": sha256(pd.util.hash_pandas_object(sample, index=False).values.tobytes()).hexdigest(),
        "max_descriptions": MAX_DESCRIPTIONS, "sample_seed": 42,
        "frequency_weighted": False,
        "silhouette_metric": "cosine", "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
        "metric_random_state": METRIC_RANDOM_STATE,
        "metric_sampling": "fixed rows, then exclude algorithmic noise",
        "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
        "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
        "silhouette_weight": SILHOUETTE_WEIGHT, "noise_weight": NOISE_WEIGHT,
        "tsne": {
            "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
            "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
            "reduction": "TruncatedSVD", "svd_components": TSNE_SVD_COMPONENTS,
            "dimensions": [2, 3], "init": "pca", "learning_rate": "auto",
            "method": "barnes_hut",
        },
    }


## 3. Carregar e preparar dados

Origem: Parquet produzido pelos notebooks de preparação de dados do projeto, com descrições de supermercado, NCMs e duplicatas agregadas por descrição. A amostragem é feita uma vez por semente. Os embeddings são gerados uma vez por amostra e reutilizados na grade de parâmetros.


In [5]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descrições disponíveis após preparação: %d", len(base_items))

# Cada execução usa uma pasta nova para preservar relatórios anteriores.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ%f")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)


2026-10-25 03:18:28,034 INFO Descrições disponíveis após preparação: 146999


## 4. Gerar embeddings, ajustar HDBSCAN e comparar parâmetros

MPNet gera embeddings densos normalizados e a distância cosseno é usada diretamente no HDBSCAN. O ruído (`cluster = -1`) é excluído da silhouette e contabilizado separadamente; configurações com menos de dois clusters válidos ficam sem silhouette. Para controlar o custo, a silhouette usa no máximo `SILHOUETTE_SAMPLE_SIZE` itens não classificados como ruído.

Para visualização, uma amostra fixa de até `TSNE_MAX_POINTS` descrições é reduzida com PCA e projetada uma vez por amostra. As coordenadas são reutilizadas para salvar um PNG por configuração.

A projeção t-SNE 3D é calculada separadamente e cada configuração também é salva como HTML interativo autocontido em `tsne_3d_html/`.


In [6]:
records = []
labels_by_run = {}
items_by_run = {}
matrices_by_representation = {}
items_by_seed = {}
vectors_by_seed = {}

logger.info("Carregando modelo %s", MODEL_NAME)
encoder = SentenceTransformer(MODEL_NAME)

for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    items_by_seed[sample_seed] = sampled_items

    logger.info("Gerando embeddings MPNet: seed=%d, descrições=%d", sample_seed, len(sampled_items))
    vectors = encoder.encode(
        sampled_items[DESCRIPTION_NORMALIZED].astype(str).tolist(),
        batch_size=ENCODE_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    vectors_by_seed[sample_seed] = vectors
    matrices_by_representation[(sample_seed, "mpnet")] = vectors

    for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
        for min_samples in MIN_SAMPLES_VALUES:
            min_samples_label = "default" if min_samples is None else str(min_samples)
            run_id = (
                f"seed{sample_seed}_mpnet_min_cluster_size{min_cluster_size}"
                f"_min_samples{min_samples_label}_{CLUSTER_SELECTION_METHOD}"
            )
            logger.info("Ajustando HDBSCAN %s", run_id)
            model = HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric="cosine",
                algorithm="brute",
                cluster_selection_method=CLUSTER_SELECTION_METHOD,
                n_jobs=HDBSCAN_N_JOBS,
                copy=True,
            )
            labels = model.fit_predict(vectors)
            metrics = score_configuration(
                vectors, labels, SILHOUETTE_SAMPLE_SIZE,
                METRIC_RANDOM_STATE,
            )
            records.append({
                "run_id": run_id, "sample_seed": sample_seed,
                "representation": "mpnet", "model_name": MODEL_NAME,
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                "cluster_selection_method": CLUSTER_SELECTION_METHOD,
                "frequency_weighted": False, **metrics,
            })
            labels_by_run[run_id] = labels
            items_by_run[run_id] = sampled_items

results = pd.DataFrame(records)


2026-10-25 03:18:28,061 INFO Carregando modelo sentence-transformers/paraphrase-multilingual-mpnet-base-v2


2026-10-25 03:18:28,063 INFO No device provided, using cpu


2026-10-25 03:18:28,087 INFO Loading SentenceTransformer model from sentence-transformers/paraphrase-multilingual-mpnet-base-v2.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-10-25 03:18:30,831 INFO Gerando embeddings MPNet: seed=42, descrições=20000


...

2026-10-25 03:24:18,476 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size10_min_samples10_eom


2026-10-25 03:24:28,084 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size20_min_samplesdefault_eom


2026-10-25 03:24:37,502 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size20_min_samples2_eom


2026-10-25 03:24:47,893 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size20_min_samples5_eom


2026-10-25 03:24:59,855 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size20_min_samples10_eom


### Calcular a metrica composta e ordenar os resultados

A silhouette e reescalada para 0 a 1 e combinada com a proporcao de descricoes classificadas (`1 - noise_fraction`). O score composto determina a ordem; silhouette e ruido servem apenas de desempate quando os scores forem iguais.


In [7]:
if not 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING:
    raise ValueError(
        "Os limites devem atender a 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING."
    )

if SILHOUETTE_WEIGHT < 0 or NOISE_WEIGHT < 0 or SILHOUETTE_WEIGHT + NOISE_WEIGHT <= 0:
    raise ValueError("Os pesos devem ser nao negativos e pelo menos um deles deve ser maior que zero.")

# Silhouette varia de -1 a 1; reescale para 0 a 1 para combinar com cobertura.
results["silhouette_quality"] = (results["silhouette_cosine_without_noise"] + 1) / 2
results["noise_quality"] = 1 - results["noise_fraction"]
weight_total = SILHOUETTE_WEIGHT + NOISE_WEIGHT
results["composite_score"] = (
    SILHOUETTE_WEIGHT * results["silhouette_quality"]
    + NOISE_WEIGHT * results["noise_quality"]
) / weight_total

# O score composto e o criterio principal; os outros criterios so desempataram scores iguais.
ranking = (
    results.loc[
        results["cluster_count_excluding_noise"].between(
            MIN_CLUSTER_FOR_RANKING, MAX_CLUSTERS_FOR_RANKING, inclusive="both",
        ) & results["silhouette_cosine_without_noise"].notna()
    ]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True], na_position="last",
    )
    .reset_index(drop=True)
)
if ranking.empty:
    raise ValueError(
        f"Nenhum resultado atende ao intervalo de {MIN_CLUSTER_FOR_RANKING} a "
        f"{MAX_CLUSTERS_FOR_RANKING} clusters. Ajuste min_cluster_size/min_samples ou o limite."
    )
# Explicita a posicao de cada configuracao depois dos filtros e criterios acima.
ranking.insert(0, "rank", range(1, len(ranking) + 1))


In [8]:
display(ranking)

,rank,run_id,sample_seed,representation,model_name,min_cluster_size,min_samples,cluster_selection_method,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_base_sample_size,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_mpnet_min_cluster_size5_min_samples10_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,10.0,eom,False,123,16619,0.83095,20000,3381,0.470984,0.735492,0.16905,0.593881
1,2,seed42_mpnet_min_cluster_size5_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,NaN,eom,False,300,15249,0.76245,20000,4751,0.420774,0.710387,0.23755,0.592178
2,3,seed42_mpnet_min_cluster_size5_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,5.0,eom,False,300,15249,0.76245,20000,4751,0.420774,0.710387,0.23755,0.592178
3,4,seed42_mpnet_min_cluster_size5_min_samples2_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,2.0,eom,False,673,12412,0.62060,20000,7588,0.325214,0.662607,0.37940,0.591805
4,5,seed42_mpnet_min_cluster_size20_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,NaN,eom,False,31,17955,0.89775,20000,2045,0.498991,0.749495,0.10225,0.587684
5,6,seed42_mpnet_min_cluster_size10_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,5.0,eom,False,147,15408,0.77040,20000,4592,0.411720,0.705860,0.22960,0.586795
6,7,seed42_mpnet_min_cluster_size10_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,NaN,eom,False,95,16710,0.83550,20000,3290,0.451607,0.725803,0.16450,0.585477
7,8,seed42_mpnet_min_cluster_size10_min_samples10_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,10.0,eom,False,95,16710,0.83550,20000,3290,0.451607,0.725803,0.16450,0.585477
8,9,seed42_mpnet_min_cluster_size20_min_samples10_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,10.0,eom,False,54,17161,0.85805,20000,2839,0.458429,0.729215,0.14195,0.582398
9,10,seed42_mpnet_min_cluster_size20_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,5.0,eom,False,74,15814,0.79070,20000,4186,0.398267,0.699133,0.20930,0.576675


In [9]:
# Exibe o ranking definido acima, sem aplicar filtros adicionais.
ranking.head(40)


,rank,run_id,sample_seed,representation,model_name,min_cluster_size,min_samples,cluster_selection_method,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_base_sample_size,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_mpnet_min_cluster_size5_min_samples10_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,10.0,eom,False,123,16619,0.83095,20000,3381,0.470984,0.735492,0.16905,0.593881
1,2,seed42_mpnet_min_cluster_size5_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,NaN,eom,False,300,15249,0.76245,20000,4751,0.420774,0.710387,0.23755,0.592178
2,3,seed42_mpnet_min_cluster_size5_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,5.0,eom,False,300,15249,0.76245,20000,4751,0.420774,0.710387,0.23755,0.592178
3,4,seed42_mpnet_min_cluster_size5_min_samples2_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,2.0,eom,False,673,12412,0.62060,20000,7588,0.325214,0.662607,0.37940,0.591805
4,5,seed42_mpnet_min_cluster_size20_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,NaN,eom,False,31,17955,0.89775,20000,2045,0.498991,0.749495,0.10225,0.587684
5,6,seed42_mpnet_min_cluster_size10_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,5.0,eom,False,147,15408,0.77040,20000,4592,0.411720,0.705860,0.22960,0.586795
6,7,seed42_mpnet_min_cluster_size10_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,NaN,eom,False,95,16710,0.83550,20000,3290,0.451607,0.725803,0.16450,0.585477
7,8,seed42_mpnet_min_cluster_size10_min_samples10_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,10.0,eom,False,95,16710,0.83550,20000,3290,0.451607,0.725803,0.16450,0.585477
8,9,seed42_mpnet_min_cluster_size20_min_samples10_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,10.0,eom,False,54,17161,0.85805,20000,2839,0.458429,0.729215,0.14195,0.582398
9,10,seed42_mpnet_min_cluster_size20_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,5.0,eom,False,74,15814,0.79070,20000,4186,0.398267,0.699133,0.20930,0.576675


## Salvar assignments de um teste específico


In [10]:
ASSIGNMENT_RUN_ID = ASSIGNMENT_RUN_ID or ranking.iloc[0]["run_id"]

run_assignments, run_cluster_summary = summarize_clusters(
    items_by_run[ASSIGNMENT_RUN_ID],
    labels_by_run[ASSIGNMENT_RUN_ID],
)

assignment_path = output_dir / f"assignments_{ASSIGNMENT_RUN_ID}.csv"
if assignment_path.exists():
    raise FileExistsError(f"Arquivo já existe: {assignment_path}")

run_assignments.to_csv(assignment_path, index=False, encoding="utf-8-sig")
print(f"Assignments salvos em: {assignment_path}")


Assignments salvos em: C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\mpnet_hdbscan\202501-202512_20261025T061828Z036879\assignments_seed42_mpnet_min_cluster_size5_min_samples10_eom.csv


## 5. Revisar um resultado e salvar relatórios

Escolha `SELECTED_RUN_ID` após comparar quantidade de clusters, ruído e silhouette. A tabela de resumo inclui `-1` como grupo de ruído. Os arquivos são gravados em uma pasta com identificador UTC único para preservar execuções anteriores.

Os gráficos ficam em `tsne_plots/`, um PNG por configuração, e as coordenadas da amostra em `tsne_coordinates/`, dentro da pasta da execução.


In [11]:
SELECTED_RUN_ID = ranking.iloc[0]["run_id"]
if SELECTED_RUN_ID not in results["run_id"].values:
    raise ValueError(f"Execucao nao encontrada em results: {SELECTED_RUN_ID}")
if SELECTED_RUN_ID not in items_by_run or SELECTED_RUN_ID not in labels_by_run:
    raise ValueError(f"Execucao sem itens ou labels carregados: {SELECTED_RUN_ID}")
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
cluster_summary = cluster_summary.sort_values("frequencia_total", ascending=False).reset_index(drop=True)
cluster_sizes = assignments.loc[assignments["cluster"].ne(-1)].groupby("cluster")[DESCRIPTION_NORMALIZED].size()
singleton_clusters = cluster_sizes[cluster_sizes.eq(1)].index
frequent_singletons = assignments[assignments["cluster"].isin(singleton_clusters)].sort_values(
    FREQUENCY, ascending=False
)
frequent_noise = assignments[assignments["cluster"].eq(-1)].sort_values(
    FREQUENCY, ascending=False
)
display(cluster_summary.head(20))
display(frequent_singletons.head(20))
display(frequent_noise.head(20))

results.to_csv(output_dir / "metrics.csv", index=False)
ranking.to_csv(output_dir / "ranking.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
frequent_singletons.to_csv(
    output_dir / "frequent_singleton_descriptions.csv", index=False, encoding="utf-8-sig"
)
frequent_noise.to_csv(
    output_dir / "frequent_noise_descriptions.csv", index=False, encoding="utf-8-sig"
)

,cluster,descricoes_unicas,frequencia_total
0,-1,16619,89411
1,63,26,1980
2,74,99,1793
3,92,209,1637
4,56,171,1512
5,121,62,1340
6,81,37,1198
7,80,82,1152
8,66,96,1076
9,85,167,979


,descricao_normalizada,frequencia,cluster,ruido


,descricao_normalizada,frequencia,cluster,ruido
15013,pilha alcalina go tech aaa lr03 palito 2un,3588,-1,True
10531,jantar,1493,-1,True
12606,milho verde,1464,-1,True
18043,salsicha,1339,-1,True
13954,pao forma,908,-1,True
14834,peito peru,907,-1,True
18070,salsicha kg,906,-1,True
13101,multiuso 500ml ye advir,871,-1,True
5507,cola bastao 40gr ref 4545,859,-1,True
1727,apagador quadro branco go office,805,-1,True


## Salvar um modelo pelo Run ID

Informe o `MODEL_RUN_ID` abaixo para exportar o encoder MPNet, o HDBSCAN ajustado, suas configurações e os assignments de treino. A exportação cria uma pasta nova em `models/mpnet_hdbscan/` para preservar modelos anteriores. O artefato HDBSCAN não fornece atribuição direta de novos pontos a clusters; o encoder é registrado pelo identificador do modelo usado.


In [12]:
MODEL_RUN_ID = MODEL_RUN_ID or ranking.iloc[0]["run_id"]

if MODEL_RUN_ID not in labels_by_run:
    raise ValueError(f"Run ID não encontrado. Opções: {list(labels_by_run)}")

model_run = results.set_index("run_id").loc[MODEL_RUN_ID]
model_items = items_by_run[MODEL_RUN_ID]
model_labels = labels_by_run[MODEL_RUN_ID]
# Reutiliza exatamente os embeddings usados no ajuste do Run ID.
model_vectors = vectors_by_seed[int(model_run["sample_seed"])]
model_min_samples = (
    None if pd.isna(model_run["min_samples"]) else int(model_run["min_samples"])
)
saved_model = HDBSCAN(
    min_cluster_size=int(model_run["min_cluster_size"]),
    min_samples=model_min_samples,
    metric="cosine",
    algorithm="brute",
    cluster_selection_method=model_run["cluster_selection_method"],
    n_jobs=HDBSCAN_N_JOBS,
    copy=True,
)
saved_model.fit(model_vectors)
if not np.array_equal(saved_model.labels_, model_labels):
    raise RuntimeError("Os labels do modelo refeito não coincidem com os assignments do Run ID.")

model_assignments, _ = summarize_clusters(model_items, model_labels)
model_output_dir = MODEL_ROOT / f"{PERIOD_LABEL}_{RUN_ID}_{MODEL_RUN_ID}"
model_output_dir.mkdir(parents=True, exist_ok=False)
encoder.save(str(model_output_dir / "sentence_transformer"))
pd.to_pickle(saved_model, model_output_dir / "hdbscan_model.pkl")
model_assignments.to_csv(
    model_output_dir / "training_assignments.csv", index=False, encoding="utf-8-sig"
)
model_config = {
    "run_id": MODEL_RUN_ID,
    "period": PERIOD_LABEL,
    "dataset_name": DATASET_NAME,
    "sample_seed": int(model_run["sample_seed"]),
    "representation": "mpnet",
    "encoder": {
        "model_name": MODEL_NAME,
        "encode_batch_size": ENCODE_BATCH_SIZE,
        "normalize_embeddings": True,
        "embedding_dimension": int(model_vectors.shape[1]),
    },
    "hdbscan": {
        "min_cluster_size": int(model_run["min_cluster_size"]),
        "min_samples": model_min_samples,
        "metric": "cosine",
        "algorithm": "brute",
        "cluster_selection_method": model_run["cluster_selection_method"],
    },
    "frequency_weighted": False,
    "artifacts": {
        "encoder": "sentence_transformer/",
        "model": "hdbscan_model.pkl",
        "training_assignments": "training_assignments.csv",
    },
}
(model_output_dir / "config.json").write_text(
    json.dumps(model_config, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Modelo do Run ID %s salvo em %s", MODEL_RUN_ID, model_output_dir)


2026-10-25 03:25:23,533 INFO Saving model to C:\Users\samuel.simao\Sources\Nofis-Classifier\models\mpnet_hdbscan\202501-202512_20261025T061828Z036879_seed42_mpnet_min_cluster_size5_min_samples10_eom\sentence_transformer


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-25 03:25:24,841 INFO Modelo do Run ID seed42_mpnet_min_cluster_size5_min_samples10_eom salvo em C:\Users\samuel.simao\Sources\Nofis-Classifier\models\mpnet_hdbscan\202501-202512_20261025T061828Z036879_seed42_mpnet_min_cluster_size5_min_samples10_eom


## 6. Gerar visualizações t-SNE

Esta etapa roda depois do ajuste e do ranking. Ela projeta os embeddings MPNet já calculados e salva as coordenadas e os gráficos, sem executar novamente o HDBSCAN.


In [13]:
# Calcula projeçãoções 2D e 3D separadas para cada amostra MPNet.
tsne_by_representation = {}
tsne_3d_by_representation = {}
for (sample_seed, representation), matrix in matrices_by_representation.items():
    logger.info("Calculando t-SNE 2D para %s, seed=%d", representation, sample_seed)
    tsne_indices, tsne_coordinates, tsne_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE,
    )
    tsne_by_representation[(sample_seed, representation)] = {
        "indices": tsne_indices, "coordinates": tsne_coordinates,
        "reduced_dimensions": tsne_dimensions,
    }

    logger.info("Calculando t-SNE 3D para %s, seed=%d", representation, sample_seed)
    tsne_3d_indices, tsne_3d_coordinates, tsne_3d_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE, n_components=TSNE_3D_N_COMPONENTS,
    )
    if not np.array_equal(tsne_indices, tsne_3d_indices):
        raise ValueError("As projeçãoções 2D e 3D devem usar a mesma amostra de descrições.")
    tsne_3d_by_representation[(sample_seed, representation)] = {
        "indices": tsne_3d_indices, "coordinates": tsne_3d_coordinates,
        "reduced_dimensions": tsne_3d_dimensions,
    }

# Salva coordenadas 2D e um PNG por configuração.
tsne_coordinates_dir = output_dir / "tsne_coordinates"
tsne_plots_dir = output_dir / "tsne_plots"
tsne_3d_html_dir = output_dir / "tsne_3d_html"
tsne_coordinates_dir.mkdir()
tsne_plots_dir.mkdir()
tsne_3d_html_dir.mkdir()
tsne_artifacts = {}
for (sample_seed, representation), projection in tsne_by_representation.items():
    indices = projection["indices"]
    coordinates = projection["coordinates"]
    coordinate_frame = items_by_seed[sample_seed].iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    coordinate_frame["tsne_1"] = coordinates[:, 0]
    coordinate_frame["tsne_2"] = coordinates[:, 1]
    coordinate_name = f"seed{sample_seed}_{representation}_coordinates.csv"
    coordinate_frame.to_csv(
        tsne_coordinates_dir / coordinate_name, index=False, encoding="utf-8-sig",
    )
    tsne_artifacts[f"seed{sample_seed}_{representation}"] = {
        "coordinates_file": f"tsne_coordinates/{coordinate_name}",
        "sample_size": len(indices),
        "reduced_dimensions": projection["reduced_dimensions"],
    }

for run_id, labels in labels_by_run.items():
    result_row = results.loc[results["run_id"].eq(run_id)].iloc[0]
    representation_key = (int(result_row["sample_seed"]), result_row["representation"])
    projection = tsne_by_representation[representation_key]
    projection_3d = tsne_3d_by_representation[representation_key]
    plot_labels = np.asarray(labels)[projection["indices"]]
    plot_path = tsne_plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], plot_labels, run_id, plot_path)
    tsne_artifacts[f"plot:{run_id}"] = str(plot_path.relative_to(output_dir))

    plot_3d_path = tsne_3d_html_dir / f"{run_id}.html"
    save_tsne_3d_html(projection_3d["coordinates"], plot_labels, run_id, plot_3d_path)
    tsne_artifacts[f"plot_3d:{run_id}"] = str(plot_3d_path.relative_to(output_dir))

logger.info("Imagens t-SNE 2D e gráficos HTML 3D salvos: %d", len(labels_by_run))

run_metadata = {
    "comparison": comparison_metadata(items_by_seed[SAMPLE_SEEDS[0]]),
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "frequency_weighted": False,
    "dataset_path": str(DATASET_PATH.relative_to(PROJECT_ROOT)), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "model_name": MODEL_NAME,
    "encode_batch_size": ENCODE_BATCH_SIZE,
    "selected_run_id": SELECTED_RUN_ID,
    "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
    "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
    "min_cluster_size_values": MIN_CLUSTER_SIZE_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
    "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
    "tsne": {
        "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
        "svd_components": TSNE_SVD_COMPONENTS,
        "three_dimensional_components": TSNE_3D_N_COMPONENTS,
        "artifacts": tsne_artifacts,
    },
}
(output_dir / "run.json").write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Relatórios e imagens t-SNE salvos em %s", output_dir)


2026-10-25 03:25:24,873 INFO Calculando t-SNE 2D para mpnet, seed=42


2026-10-25 03:25:38,321 INFO Calculando t-SNE 3D para mpnet, seed=42


2026-10-25 03:26:08,334 INFO Imagens t-SNE 2D e gráficos HTML 3D salvos: 16


2026-10-25 03:26:08,351 INFO Relatórios e imagens t-SNE salvos em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\mpnet_hdbscan\202501-202512_20261025T061828Z036879


## 7. Interpretação

- `noise_fraction` alta significa que muitas descrições não entraram em grupos densos para os parâmetros escolhidos.
- `min_cluster_size` controla o menor grupo selecionável. `min_samples` maior tende a exigir evidência local mais forte e pode aumentar o ruído.
- Silhouette mais alta deve ser lida junto da fração de ruído e do tamanho da amostra usada para calculá-la; uma pontuação calculada em poucos itens pode ser enganosa.
- HDBSCAN não atribui automaticamente novas descrições a clusters neste fluxo e não define categorias sem revisão humana.
- MPNet representa similaridade semântica; isso pode agrupar descrições com vocabulário diferente, mas não substitui a revisão humana nem garante coerência fiscal.

Os HTML em `tsne_3d_html/` contêm uma projeção t-SNE 3D que pode ser girada com o mouse e aberta sem dependências externas.
